# Feature Engineering

In [1]:
import pandas as pd

df = pd.read_csv('data/ai4i2020.csv')
df.head(3)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


In [2]:
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df[['Air temperature [K]', 'Process temperature [K]', 'temp_diff']].head()

,Air temperature [K],Process temperature [K],temp_diff
0,298.1,308.6,10.5
1,298.2,308.7,10.5
2,298.1,308.5,10.4
3,298.2,308.6,10.4
4,298.2,308.7,10.5


In [3]:
hdf_rule = (df['temp_diff'] < 8.6) & (df['Rotational speed [rpm]'] < 1380)
pd.crosstab(hdf_rule, df['HDF'])

HDF,0,1
row_0,,
False,9885,0
True,0,115


In [4]:
import numpy as np

df['power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60
df['power'].describe()

count    10000.000000
mean      6279.744953
std       1067.418295
min       1148.440610
25%       5561.184484
50%       6271.027344
75%       7003.002724
max      10469.923005
Name: power, dtype: float64

In [5]:
pwf_rule = (df['power'] < 3500) | (df['power'] > 9000)
pd.crosstab(pwf_rule, df['PWF'])

PWF,0,1
power,,
False,9905,0
True,0,95


In [6]:
df['wear_torque'] = df['Tool wear [min]'] * df['Torque [Nm]']

limits = df['Type'].map({'L': 11000, 'M': 12000, 'H': 13000})
osf_rule = df['wear_torque'] > limits
pd.crosstab(osf_rule, df['OSF'])

OSF,0,1
row_0,,
False,9902,0
True,0,98


In [7]:
any_rule = hdf_rule | pwf_rule | osf_rule
pd.crosstab(any_rule, df['Machine failure'])

Machine failure,0,1
row_0,,
False,9661,52
True,0,287


In [8]:
modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
unexplained = (~any_rule) & (df['Machine failure'] == 1)

print(df[unexplained][modes].sum())
print("No mode recorded:", (df[unexplained][modes].sum(axis=1) == 0).sum())

TWF    43
HDF     0
PWF     0
OSF     0
RNF     1
dtype: int64
No mode recorded: 9


In [9]:
twf_wear = df[df['TWF'] == 1]['Tool wear [min]']
print(twf_wear.describe())
print("Rows with tool wear 200-240:", df['Tool wear [min]'].between(200, 240).sum())

count     46.000000
mean     216.369565
std       12.257168
min      198.000000
25%      207.000000
50%      214.500000
75%      224.250000
max      253.000000
Name: Tool wear [min], dtype: float64
Rows with tool wear 200-240: 790


What did we do above?
We created three new columns to describe the measurements more precisely: the temperature difference between the process and the air, the power, and the product of tool wear and torque.

Then we converted the units so that power comes out in watts.

Then we tested the new columns on the data, and the results matched 100%. No healthy machine met any of the rules. So we compare the models against the results of the three rules: if a rule applies, the machine is considered faulty, and if it does not, we consider it healthy (although some failures were missed by the rules).

Most of the failures the rules missed are due to tool wear, because the tool is replaced or fails at a random time between 200 and 240 minutes. This means tool wear alone is not enough to predict failure, because the actual failure time is random and is not in the columns.

In [10]:
df.to_csv('data/ai4i2020_features.csv', index=False)